In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

datos = {"naranjas": [2, 3, 1, 0], "manzanas": [0, 3, 7, 2]}

compras = pd.DataFrame(datos)
compras

In [ ]:
url = "https://raw.githubusercontent.com/ManarOmar/New-York-Airbnb-2019/master/AB_NYC_2019.csv"
data = pd.read_csv(url)
data.head()

In [ ]:
print("Dimensiones (filas, columnas):", data.shape)
data.info()
print("¿La columna 'id' es única?:", data["id"].is_unique)
print("Total de registros:", len(data))
print("Hosts (anfitriones) únicos:", data["host_id"].nunique())

In [ ]:
data.describe()

In [ ]:
columnas = [
    "price", "minimum_nights", "number_of_reviews",
    "reviews_per_month", "calculated_host_listings_count", "availability_365"
]
data[columnas].describe()

In [ ]:
data.isnull().sum()

In [ ]:
# Anuncios sin reseñas
sin_resenas = data["number_of_reviews"] == 0
print("Anuncios sin reseñas:", sin_resenas.sum())

print("Faltantes en reviews_per_month:", data["reviews_per_month"].isnull().sum())
# Confirmamos que los faltantes coinciden con anuncios sin reseñas
print("Faltantes que son exactamente anuncios sin reseñas:",
      (data["reviews_per_month"].isnull() & sin_resenas).sum())

In [ ]:
# Distribuciones de las variables numéricas
data.hist(bins=10, figsize=(10, 6))
plt.tight_layout()
plt.show()

In [ ]:
# Categóricas: cuántos hay de cada tipo
data["neighbourhood"].value_counts()

In [ ]:
data["room_type"].value_counts()

In [ ]:
data["neighbourhood_group"].value_counts()

In [ ]:
# Calcular media de reviews_per_month (solo como referencia)
media = data["reviews_per_month"].mean()
print("Media de reviews_per_month:", round(media, 2))

# Imputamos con 0: un anuncio sin reseñas tiene 0 reseñas por mes
data["reviews_per_month"] = data["reviews_per_month"].fillna(0)
print("Faltantes restantes en reviews_per_month:", data["reviews_per_month"].isnull().sum())

In [ ]:
numericas = data.select_dtypes(include=np.number)
numericas.columns.tolist()

In [ ]:
# Veamos los casos extremos con BoxPlot
data.boxplot(column="price", figsize=(8, 10))
plt.ylabel("Precio por noche (USD)")
plt.show()

In [ ]:
# Definimos límites razonables para el precio y filtramos
precio_minimo = 0
precio_maximo = 500  # ajustable según el criterio de outliers deseado

datos_limpios = data[(data["price"] > precio_minimo) & (data["price"] <= precio_maximo)].copy()

print("Antes:", len(data), "anuncios")
print("Después:", len(datos_limpios), "anuncios")
datos_limpios["price"].describe()

In [ ]:
# Veamos los casos extremos con BoxPlot, ya filtrados
datos_limpios.boxplot(column="price", figsize=(8, 10))
plt.ylabel("Precio por noche (USD)")
plt.show()

In [ ]:
# El histograma de precio ahora sí se lee
datos_limpios["price"].hist(bins=50, figsize=(10, 5))
plt.xlabel("Precio por noche (USD)")
plt.ylabel("Anuncios")
plt.title("Distribución del precio")
plt.show()

In [ ]:
corte_economico = 80
corte_medio = 200

def categoria_precio(precio):
    if precio <= corte_economico:
        return "económico"
    elif precio <= corte_medio:
        return "medio"
    else:
        return "premium"

datos_limpios["categoria"] = datos_limpios["price"].apply(categoria_precio)
datos_limpios["categoria"].value_counts()

In [ ]:
# Ejercicio: cortes alternativos (60 y 150)
def categoria_precio_alt(precio, corte_bajo=60, corte_alto=150):
    if precio <= corte_bajo:
        return "económico"
    elif precio <= corte_alto:
        return "medio"
    else:
        return "premium"

datos_limpios["categoria_alt"] = datos_limpios["price"].apply(categoria_precio_alt)
datos_limpios["categoria_alt"].value_counts()

# Con cortes más bajos, menos anuncios caen en "económico" y más pasan a "medio"/"premium":
# la categoría "económico" se vuelve más exigente (más pequeña) y "premium" crece.

In [ ]:
datos_limpios.groupby("neighbourhood_group")["price"].agg(["count", "mean", "median"])

In [ ]:
datos_limpios.groupby(["neighbourhood_group", "room_type"])["price"].median().unstack()

In [ ]:
datos_limpios.boxplot(column="price", by="room_type", figsize=(10, 6))
plt.xlabel("Tipo de habitación")
plt.ylabel("Precio por noche (USD)")
plt.title("Precio por tipo de habitación")
plt.suptitle("")
plt.show()

In [ ]:
datos_limpios.groupby("neighbourhood_group")["price"].median().sort_values().plot(kind="barh", color="steelblue")
plt.xlabel("Precio mediano por noche (USD)")
plt.ylabel("")
plt.title("Precio mediano por distrito")
plt.show()

In [ ]:
datos_limpios.corr(numeric_only=True)